# Aurora Probability Calibration

*Reproducible research notebook from [Flarient](https://flarient.com) — the space weather intelligence platform.*

**About this notebook:** This notebook is part of the [Flarient Research Notebooks](https://github.com/flarientglobal/flarient-notebooks) collection. It uses public data from NOAA SWPC, NASA, and the Flarient API.


## 1. Introduction

Aurora visibility depends on geomagnetic activity (Kp), your latitude, and local conditions. In this notebook, we'll calibrate an aurora probability model using public data.


In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Aurora Oval Model

The auroral oval expands equatorward as Kp increases. We'll model the southern boundary of the aurora oval as a function of Kp.


In [ ]:
# Aurora oval southern boundary model
# Based on the relationship: aurora_lat ≈ 67 - (Kp - 2) * 3.5 (for Kp >= 2)
def aurora_boundary(kp):
    if kp < 2:
        return 67.0
    elif kp >= 9:
        return 40.0
    else:
        return 67.0 - (kp - 2) * 3.5

# Generate the model curve
kp_range = np.linspace(0, 9, 100)
aurora_lat = [aurora_boundary(k) for k in kp_range]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(kp_range, aurora_lat, color='#22d3ee', linewidth=2)
ax.fill_between(kp_range, aurora_lat, 90, alpha=0.15, color='#22d3ee', label='Aurora visible')
ax.set_xlabel('Kp Index')
ax.set_ylabel('Lowest Aurora Latitude (°)')
ax.set_title('Aurora Oval Southern Boundary vs Kp — Model')
ax.invert_yaxis()
ax.grid(True, alpha=0.2)
ax.legend()
plt.tight_layout()
plt.savefig('aurora_oval_model.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Probability Calculation

For a given location and Kp, we can estimate the probability of seeing aurora.


In [ ]:
def aurora_probability(observer_lat, kp, bz=None, speed=None):
    """Calculate aurora visibility probability."""
    boundary = aurora_boundary(kp)
    lat_diff = abs(abs(observer_lat) - boundary)
    
    if lat_diff <= 0:
        prob = 95
    elif lat_diff <= 2:
        prob = 85
    elif lat_diff <= 5:
        prob = 70
    elif lat_diff <= 8:
        prob = 50
    elif lat_diff <= 12:
        prob = 30
    elif lat_diff <= 15:
        prob = 15
    elif lat_diff <= 20:
        prob = 5
    else:
        prob = 0
    
    # Bz adjustment
    if bz is not None and bz < -5:
        prob = min(95, prob + 10)
    if speed is not None and speed > 500:
        prob = min(95, prob + 5)
    
    return max(0, min(95, int(prob)))

# Example: Edinburgh (55.95°N)
print("Aurora probability for Edinburgh (55.95°N):")
for kp in [2, 3, 4, 5, 6, 7, 8, 9]:
    prob = aurora_probability(55.95, kp)
    print(f"  Kp {kp}: {prob}%")


## 4. Fetching Live Data

Let's fetch current Kp and calculate aurora probability for several cities.


In [ ]:
# Fetch current Kp
kp_resp = requests.get("https://services.swpc.noaa.gov/json/planetary_k_index_1m.json", timeout=30)
kp_data = kp_resp.json()
current_kp = float(kp_data[-1]['kp'])

# Cities to check
cities = [
    ("Edinburgh", 55.95), ("London", 51.51), ("Tromsø", 69.65),
    ("Reykjavik", 64.13), ("Anchorage", 61.22), ("Seattle", 47.61),
    ("Minneapolis", 44.98), ("New York", 40.71),
]

print(f"Current Kp: {current_kp}")
print(f"{'City':<15} {'Latitude':>8} {'Probability':>12}")
print("-" * 40)
for name, lat in cities:
    prob = aurora_probability(lat, current_kp)
    print(f"{name:<15} {lat:>8.2f} {prob:>11}%")


## 5. Probability Map

Let's visualise aurora probability across latitudes for different Kp levels.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
latitudes = np.linspace(40, 75, 100)

for kp, color in [(3, '#6366f1'), (5, '#f59e0b'), (7, '#ef4444'), (9, '#22c55e')]:
    probs = [aurora_probability(lat, kp) for lat in latitudes]
    ax.plot(latitudes, probs, color=color, linewidth=2, label=f'Kp {kp}')

ax.set_xlabel('Observer Latitude (°)')
ax.set_ylabel('Aurora Probability (%)')
ax.set_title('Aurora Probability by Latitude and Kp — Model')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('aurora_probability_map.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Conclusion

This notebook showed:
1. How the aurora oval expands with increasing Kp
2. A probability model for aurora visibility at any latitude
3. Live probability calculations for major cities

For live aurora badges for your city, visit the [Flarient Aurora Badges](https://github.com/flarientglobal/flarient-aurora-badges) project. For detailed forecasts, visit [flarient.com](https://flarient.com).


---

## About Flarient

[Flarient](https://flarient.com) is a space weather intelligence platform providing real-time data, forecasts, and community-driven observations. Visit [flarient.com](https://flarient.com) for live space weather conditions, aurora forecasts, and more.

## License

MIT — This notebook is open source. [View on GitHub](https://github.com/flarientglobal/flarient-notebooks).
